# 002 · SQLite Transactions and Row Factories

> **Standard Library › Persistence** · Python 3.14

Once the basics work, real code needs three more things: **transactions** that commit or roll back as a unit, **rows you can access by name** (or map to objects, like Dapper), and **types** such as `datetime` that SQLite doesn't know natively. This notebook covers all three and ends with a small repository class.

**In this notebook you will learn:**

- how `with conn:` commits or rolls back — and why it does **not** close the connection
- the legacy `isolation_level` behaviour vs the `autocommit` attribute (3.12+)
- `row_factory`: `sqlite3.Row`, dict rows and dataclass rows
- registering adapters and converters for `datetime` (the defaults are deprecated since 3.12)
- handling `IntegrityError`, and wrapping it all in a repository

## .NET mental model

| .NET | Python `sqlite3` |
|---|---|
| `using var tx = conn.BeginTransaction(); ... tx.Commit();` | `with conn: ...` (commit on success, rollback on exception) |
| `TransactionScope` | — (no ambient transactions) |
| `reader["Name"]` | `row["name"]` with `row_factory = sqlite3.Row` |
| Dapper `conn.Query<Product>(sql)` | a `row_factory` that builds a dataclass |
| Dapper `SqlMapper.AddTypeHandler<T>` / EF value converters | `sqlite3.register_adapter` / `register_converter` |
| `SqliteException` with `SqliteErrorCode == 19` (constraint) | `sqlite3.IntegrityError` |

In [ ]:
import sqlite3
from contextlib import closing

conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE account (id INTEGER PRIMARY KEY, owner TEXT UNIQUE NOT NULL, balance INTEGER NOT NULL CHECK (balance >= 0))")
conn.executemany("INSERT INTO account (owner, balance) VALUES (?, ?)", [("alice", 100), ("bob", 50)])
conn.commit()

def balances(c=conn):
    return c.execute("SELECT owner, balance FROM account ORDER BY owner").fetchall()

print(balances())

## The connection as a context manager

`with conn:` is Python's version of a transaction block:

- if the block finishes normally → `conn.commit()`
- if it raises → `conn.rollback()`, and the exception propagates

```csharp
// C#
using var tx = conn.BeginTransaction();
Debit(conn, "alice", 30);
Credit(conn, "bob", 30);
tx.Commit();               // disposing without Commit() rolls back
```

In [ ]:
def transfer(conn, source, target, amount):
    with conn:                                   # one transaction
        conn.execute("UPDATE account SET balance = balance - ? WHERE owner = ?", (amount, source))
        conn.execute("UPDATE account SET balance = balance + ? WHERE owner = ?", (amount, target))

transfer(conn, "alice", "bob", 30)
print(balances())

try:
    transfer(conn, "bob", "alice", 500)          # violates CHECK (balance >= 0)
except sqlite3.IntegrityError as e:
    print(f"{type(e).__name__}: {e}")
print(balances())                                # unchanged — the whole block rolled back

⚠️ **Gotcha:** unlike a C# `using` on a connection, `with conn:` **does not close** the connection. The connection is still open after the block; close it yourself, or nest it with `contextlib.closing` when you want both.

In [ ]:
with conn:
    pass
print(conn.execute("SELECT 'still open'").fetchone()[0])

with closing(sqlite3.connect(":memory:")) as c, c:     # close AND transaction
    c.execute("CREATE TABLE t (x)")
    c.execute("INSERT INTO t VALUES (1)")
try:
    c.execute("SELECT 1")
except sqlite3.ProgrammingError as e:
    print(f"{type(e).__name__}: {e}")

## How transactions start: legacy `isolation_level`

By default `sqlite3` uses its historical ("legacy") transaction control, governed by `isolation_level` (default `""`, meaning `DEFERRED`):

- an implicit `BEGIN` is issued **before `INSERT`, `UPDATE`, `DELETE` or `REPLACE`**
- other statements (`SELECT`, `CREATE TABLE`, ...) do **not** open a transaction
- nothing is committed until you call `commit()` (or leave a `with conn:` block); `close()` without commit **discards** pending changes
- `isolation_level=None` turns implicit transactions off: every statement commits immediately (SQLite's own autocommit), and you write `BEGIN`/`COMMIT` yourself

`conn.in_transaction` tells you whether a transaction is open right now.

In [ ]:
print(repr(conn.isolation_level), conn.autocommit == sqlite3.LEGACY_TRANSACTION_CONTROL)

print(conn.in_transaction)                       # False
conn.execute("SELECT * FROM account").fetchall()
print(conn.in_transaction)                       # False — SELECT doesn't begin
conn.execute("UPDATE account SET balance = balance + 1 WHERE owner = 'alice'")
print(conn.in_transaction)                       # True  — implicit BEGIN
conn.rollback()
print(conn.in_transaction, balances())           # False, change undone

## The `autocommit` attribute (Python 3.12+)

Python 3.12 added a clearer, PEP 249-compliant switch: `sqlite3.connect(..., autocommit=...)`. When it is set to `True` or `False`, `isolation_level` is ignored.

| `autocommit` | Behaviour |
|---|---|
| `sqlite3.LEGACY_TRANSACTION_CONTROL` (default) | the legacy `isolation_level` rules above |
| `False` (recommended for new code) | a transaction is **always** open; `commit()`/`rollback()` immediately start a new one; `close()` rolls back uncommitted work |
| `True` | SQLite autocommit: every statement is its own transaction; `commit()`/`rollback()` do nothing — use explicit `BEGIN`/`COMMIT` SQL |

The documentation says the default will change to `False` in a future Python version, so setting it explicitly makes your code future-proof.

In [ ]:
with closing(sqlite3.connect(":memory:", autocommit=False)) as c:
    print("open at start:", c.in_transaction)        # True — even before any statement
    c.execute("CREATE TABLE t (x)")
    c.execute("INSERT INTO t VALUES (1)")
    c.commit()
    print("open after commit:", c.in_transaction)    # True — a new one began
    c.execute("INSERT INTO t VALUES (2)")
    c.rollback()
    print(c.execute("SELECT x FROM t").fetchall())   # [(1,)]

⚠️ **Gotcha:** with `autocommit=True`, `with conn:` and `conn.rollback()` do **nothing** — nothing is rolled back on error. You must manage transactions with SQL:

In [ ]:
with closing(sqlite3.connect(":memory:", autocommit=True)) as c:
    c.execute("CREATE TABLE t (x)")
    try:
        with c:
            c.execute("INSERT INTO t VALUES (1)")
            raise RuntimeError("boom")
    except RuntimeError:
        pass
    print(c.execute("SELECT x FROM t").fetchall())   # [(1,)] — NOT rolled back!

    c.execute("BEGIN")
    c.execute("INSERT INTO t VALUES (2)")
    c.execute("ROLLBACK")                            # explicit SQL works
    print(c.execute("SELECT x FROM t").fetchall())

**Tip:** for a write that must not be interleaved with other writers (e.g. read-then-update), `BEGIN IMMEDIATE` takes the write lock up front — set `isolation_level="IMMEDIATE"` in legacy mode, or issue it yourself with `autocommit=True`.

## Row factories: access columns by name

By default rows are plain tuples. Set `conn.row_factory` to change what each row becomes. The built-in `sqlite3.Row` is the quick win: index by position **or** by (case-insensitive) column name, `keys()`, and easy conversion to `dict`.

```csharp
// C#
using var reader = cmd.ExecuteReader();
while (reader.Read())
    Console.WriteLine($"{reader["owner"]}: {reader.GetInt32(reader.GetOrdinal("balance"))}");
```

In [ ]:
conn.row_factory = sqlite3.Row

row = conn.execute("SELECT id, owner, balance FROM account WHERE owner = ?", ("alice",)).fetchone()
print(type(row).__name__)
print(row["owner"], row["BALANCE"], row[0])   # name (any case) or index
print(row.keys())
print(dict(row))

## Dict and dataclass rows (Dapper-style mapping)

A row factory is any callable `factory(cursor, row_tuple)`. Use `cursor.description` for the column names. A dataclass factory gives you something very close to Dapper's `Query<T>`:

```csharp
// C# with Dapper
public record Account(long Id, string Owner, long Balance);
var accounts = conn.Query<Account>("SELECT id, owner, balance FROM account").ToList();
```

In [ ]:
from dataclasses import dataclass

def dict_factory(cursor, row):
    return {col[0]: value for col, value in zip(cursor.description, row)}

@dataclass(frozen=True)
class Account:
    id: int
    owner: str
    balance: int

def dataclass_factory(cls):
    def factory(cursor, row):
        return cls(**{col[0]: value for col, value in zip(cursor.description, row)})
    return factory

conn.row_factory = dict_factory
print(conn.execute("SELECT owner, balance FROM account ORDER BY owner").fetchall())

cur = conn.cursor()
cur.row_factory = dataclass_factory(Account)    # can be set per cursor, too
print(cur.execute("SELECT id, owner, balance FROM account ORDER BY owner").fetchall())

conn.row_factory = None                          # back to tuples

The column names in the query must match the dataclass fields — use `AS` aliases where they differ (just as with Dapper).

## Dates and times: adapters and converters

SQLite has no date type; dates are usually stored as ISO-8601 `TEXT`. Python used to ship default adapters for `date`/`datetime`, but they are **deprecated since Python 3.12** (they also ignore time zones). Using them now emits a `DeprecationWarning`:

In [ ]:
import warnings
from datetime import date, datetime, UTC

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    conn.execute("SELECT ?", (datetime(2025, 3, 1, 12, 30),))
for w in caught:
    print(f"{w.category.__name__}: {w.message}")

The recommended fix is to register your own:

- an **adapter** converts a Python object → a SQLite value when binding parameters (`register_adapter(type, fn)`)
- a **converter** converts the stored `bytes` → a Python object when reading, chosen by the declared column type (`register_converter("TYPENAME", fn)`), and only if you connect with `detect_types=sqlite3.PARSE_DECLTYPES`

⚠️ **Gotcha:** both registries are **global** for the process — like a static type-handler registration in Dapper.

In [ ]:
def adapt_datetime(value: datetime) -> str:
    return value.isoformat()                              # keeps the UTC offset if present

def convert_datetime(raw: bytes) -> datetime:
    return datetime.fromisoformat(raw.decode())

sqlite3.register_adapter(datetime, adapt_datetime)
sqlite3.register_adapter(date, date.isoformat)
sqlite3.register_converter("timestamp", convert_datetime)  # column type names are case-insensitive
sqlite3.register_converter("date", lambda raw: date.fromisoformat(raw.decode()))

In [ ]:
with closing(sqlite3.connect(":memory:", detect_types=sqlite3.PARSE_DECLTYPES)) as c:
    c.execute("CREATE TABLE event (name TEXT, at TIMESTAMP, day DATE)")
    c.execute("INSERT INTO event VALUES (?, ?, ?)",
              ("deploy", datetime(2025, 3, 1, 12, 30, tzinfo=UTC), date(2025, 3, 1)))

    print(c.execute("SELECT at, typeof(at) FROM event").fetchone()[1])    # stored as text
    name, at, day = c.execute("SELECT name, at, day FROM event").fetchone()
    print(repr(at))
    print(repr(day))

Without `detect_types`, or for computed expressions, you just get the string back. `PARSE_COLNAMES` lets a query request a converter with an alias like `AS "when [timestamp]"`:

In [ ]:
with closing(sqlite3.connect(":memory:", detect_types=sqlite3.PARSE_COLNAMES)) as c:
    raw = c.execute("SELECT ?", (datetime(2025, 3, 1, 8, 0),)).fetchone()[0]
    typed = c.execute('SELECT ? AS "when [timestamp]"', (datetime(2025, 3, 1, 8, 0),)).fetchone()[0]
    print(repr(raw))
    print(repr(typed))

**Tip:** the same mechanism works for `Decimal` (store as `TEXT`), `UUID`, enums, or JSON columns.

## Handling `IntegrityError`

Constraint violations (`UNIQUE`, `NOT NULL`, `CHECK`, `FOREIGN KEY`) raise `sqlite3.IntegrityError`. Since 3.11, `sqlite3` exceptions carry the extended SQLite error code in `sqlite_errorcode` / `sqlite_errorname`.

⚠️ **Gotcha:** SQLite does **not** enforce foreign keys unless you run `PRAGMA foreign_keys = ON` on each connection.

In [ ]:
try:
    with conn:
        conn.execute("INSERT INTO account (owner, balance) VALUES (?, ?)", ("alice", 10))
except sqlite3.IntegrityError as e:
    print(f"{type(e).__name__}: {e}")
    print(e.sqlite_errorname)

print(conn.execute("PRAGMA foreign_keys").fetchone()[0])    # 0 = off by default

## A tiny repository

Putting it together: a repository class that owns its connection, uses dataclass rows, wraps writes in transactions, and translates `IntegrityError` into a domain exception — roughly what you'd write with Dapper in C#.

In [ ]:
from decimal import Decimal

@dataclass(frozen=True)
class Product:
    id: int
    sku: str
    name: str
    price: Decimal
    created: datetime

class DuplicateSkuError(Exception):
    pass

sqlite3.register_adapter(Decimal, str)
sqlite3.register_converter("decimal", lambda raw: Decimal(raw.decode()))

class ProductRepository:
    def __init__(self, path: str = ":memory:"):
        self._conn = sqlite3.connect(path, detect_types=sqlite3.PARSE_DECLTYPES, autocommit=False)
        self._conn.row_factory = dataclass_factory(Product)
        self._conn.execute("""CREATE TABLE IF NOT EXISTS product (
            id INTEGER PRIMARY KEY, sku TEXT UNIQUE NOT NULL, name TEXT NOT NULL,
            price DECIMAL TEXT NOT NULL, created TIMESTAMP NOT NULL)""")
        self._conn.commit()

    def add(self, sku: str, name: str, price: Decimal) -> Product:
        try:
            with self._conn:
                return self._conn.execute(
                    "INSERT INTO product (sku, name, price, created) VALUES (?, ?, ?, ?) RETURNING *",
                    (sku, name, price, datetime(2025, 3, 1, tzinfo=UTC)),
                ).fetchone()
        except sqlite3.IntegrityError as e:
            raise DuplicateSkuError(sku) from e

    def get(self, sku: str) -> Product | None:
        return self._conn.execute("SELECT * FROM product WHERE sku = ?", (sku,)).fetchone()

    def list_cheaper_than(self, limit: Decimal) -> list[Product]:
        rows = self._conn.execute("SELECT * FROM product ORDER BY sku").fetchall()
        return [p for p in rows if p.price < limit]    # compare as Decimal, not TEXT

    def close(self) -> None:
        self._conn.close()

⚠️ **Gotcha:** SQLite's *type affinity* is derived from the declared type name. A column declared just `DECIMAL` gets NUMERIC affinity, which would silently turn the text `'0.50'` into the REAL `0.5` — losing exactness. Declaring it `DECIMAL TEXT` gives TEXT affinity (the name contains `TEXT`), while `PARSE_DECLTYPES` still picks the converter from the **first word**, `DECIMAL`. The price of storing text is that SQL comparisons like `price < 5` would compare text with numbers — so `list_cheaper_than` filters in Python. In a real app, consider storing money as integer cents instead.

In [ ]:
repo = ProductRepository()
print(repo.add("A1", "apple", Decimal("0.50")))
repo.add("B2", "melon", Decimal("2.10"))
repo.add("C3", "hammer", Decimal("12.00"))

try:
    repo.add("A1", "another apple", Decimal("0.60"))
except DuplicateSkuError as e:
    print(f"{type(e).__name__}: {e} (caused by {type(e.__cause__).__name__})")

print(repo.get("B2"))
print(repo.get("ZZ"))
print([p.sku for p in repo.list_cheaper_than(Decimal("5"))])
repo.close()

In [ ]:
conn.close()

## Summary

| Task | C# / .NET | Python `sqlite3` |
|---|---|---|
| Transaction block | `using var tx = conn.BeginTransaction(); tx.Commit();` | `with conn:` |
| Also close | `using var conn = ...` | `with closing(conn), conn:` |
| Explicit commit / rollback | `tx.Commit()` / `tx.Rollback()` | `conn.commit()` / `conn.rollback()` |
| Is a transaction open? | `db.Database.CurrentTransaction != null` (EF Core) | `conn.in_transaction` |
| Modern transaction mode | — | `connect(..., autocommit=False)` (3.12+) |
| Column by name | `reader["owner"]` | `row["owner"]` with `sqlite3.Row` |
| Map rows to objects | Dapper `Query<T>()` | custom `row_factory` |
| Custom type in | Dapper type handler `SetValue` | `sqlite3.register_adapter(T, fn)` |
| Custom type out | Dapper type handler `Parse` | `register_converter("TYPE", fn)` + `detect_types` |
| Constraint violation | `SqliteException` (code 19) | `sqlite3.IntegrityError` |

## Exercises

1. Using `autocommit=False`, insert two rows, `commit()`, insert a third, then `close()` the connection *without* committing. Reopen the database file (inside a `TemporaryDirectory`) and show that only two rows exist.
2. Write a `namedtuple_factory` row factory that returns `collections.namedtuple` instances whose fields are the column names, and use it to print `row.owner` for each account.
3. Register an adapter/converter pair that stores a `set[str]` of tags as a comma-separated `TEXT` column declared as `TAGS`, and round-trip `{"red", "sale"}` through a table.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
import sqlite3
import tempfile
from collections import namedtuple
from contextlib import closing
from pathlib import Path

# 1 — close() rolls back uncommitted work in autocommit=False mode
with tempfile.TemporaryDirectory() as tmp:
    db = Path(tmp) / "test.db"
    c = sqlite3.connect(db, autocommit=False)
    c.execute("CREATE TABLE t (x)")
    c.executemany("INSERT INTO t VALUES (?)", [(1,), (2,)])
    c.commit()
    c.execute("INSERT INTO t VALUES (3)")
    c.close()
    with closing(sqlite3.connect(db)) as c2:
        print(c2.execute("SELECT COUNT(*) FROM t").fetchone()[0])   # 2

# 2 — build the namedtuple type from cursor.description
def namedtuple_factory(cursor, row):
    Row = namedtuple("Row", [col[0] for col in cursor.description])
    return Row(*row)

with closing(sqlite3.connect(":memory:")) as c:
    c.row_factory = namedtuple_factory
    c.execute("CREATE TABLE account (owner TEXT, balance INTEGER)")
    c.executemany("INSERT INTO account VALUES (?, ?)", [("alice", 70), ("bob", 80)])
    for row in c.execute("SELECT owner, balance FROM account ORDER BY owner"):
        print(row.owner, row.balance)

# 3 — adapter for the Python type, converter for the declared column type
sqlite3.register_adapter(set, lambda tags: ",".join(sorted(tags)))
sqlite3.register_converter("TAGS", lambda raw: set(raw.decode().split(",")) if raw else set())

with closing(sqlite3.connect(":memory:", detect_types=sqlite3.PARSE_DECLTYPES)) as c:
    c.execute("CREATE TABLE item (name TEXT, tags TAGS)")
    c.execute("INSERT INTO item VALUES (?, ?)", ("shirt", {"red", "sale"}))
    tags = c.execute("SELECT tags FROM item").fetchone()[0]
    print(sorted(tags), tags == {"red", "sale"})   # ['red', 'sale'] True
```

</details>